# BTC Liquidation Heatmap

Visualize Bitcoin liquidation events on Hyperliquid using the [0xArchive SDK](https://pypi.org/project/oxarchive/).

This notebook fetches historical liquidation data and creates:
- **Price-level heatmap** showing where liquidations cluster
- **Time-of-day analysis** revealing when liquidations spike
- **Long vs Short breakdown** with volume comparison
- **Liquidation cascade detection** identifying chain reactions

**Requirements:** Free tier API key from [0xarchive.io/dashboard](https://0xarchive.io/dashboard) (BTC, 30-day lookback)

## 1. Setup

In [ ]:
%pip install oxarchive pandas matplotlib seaborn numpy python-dotenv -q

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from oxarchive import Client

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# --- Configuration ---
# Load API key from .env file (copy .env.example to .env and add your key)
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

API_KEY = os.environ.get("OXARCHIVE_API_KEY", "your_api_key_here")
if API_KEY == "your_api_key_here":
    raise ValueError("Set OXARCHIVE_API_KEY in .env or as an environment variable")

COIN = "BTC"
LOOKBACK_DAYS = 3  # Increase for more data (max 30 on Free: history covers the most recent rolling 30 days)

client = Client(api_key=API_KEY)
end = datetime.now(timezone.utc)
start = end - timedelta(days=LOOKBACK_DAYS)
print(f"Fetching {COIN} liquidations: {start:%Y-%m-%d %H:%M} -> {end:%Y-%m-%d %H:%M} UTC")

## 2. Fetch Liquidation Data

The SDK handles cursor-based pagination automatically. We collect all liquidation events in the window.

In [ ]:
records = []
result = client.hyperliquid.liquidations.history(COIN, start=start, end=end, limit=1000)
records.extend(result.data)

while result.next_cursor:
    result = client.hyperliquid.liquidations.history(
        COIN, start=start, end=end, cursor=result.next_cursor, limit=1000
    )
    records.extend(result.data)
    print(f"\rFetched {len(records):,} liquidations...", end="", flush=True)

print(f"\rTotal: {len(records):,} liquidations in {LOOKBACK_DAYS} days")

In [ ]:
# Each trade_id appears twice in the API response (both sides of the fill).
# Filter to "Close" direction rows to get only the liquidated positions,
# then deduplicate by trade_id.
df = pd.DataFrame(
    [
        {
            "trade_id": r.trade_id,
            "timestamp": r.timestamp,
            "price": float(r.price),
            "size": float(r.size),
            "direction": r.direction,
            "mark_price": float(r.mark_price) if r.mark_price else None,
            "notional": float(r.price) * float(r.size),
        }
        for r in records
        if r.direction and r.direction.startswith("Close")
    ]
)
df = df.drop_duplicates(subset=["trade_id"], keep="first")

# Map direction to readable side label
df["side"] = df["direction"].map({"Close Long": "Long", "Close Short": "Short"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["hour"] = df["timestamp"].dt.hour
df["date"] = df["timestamp"].dt.date
df = df.drop(columns=["trade_id", "direction"])

print(f"DataFrame shape: {df.shape}")
print(f"Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
print(f"Total notional liquidated: ${df['notional'].sum():,.0f}")
df.head()

## 3. Fetch Price Data (for overlay)

Pull hourly candles to overlay the BTC price on our liquidation charts.

In [ ]:
candle_records = []
candle_result = client.hyperliquid.candles.history(
    COIN, start=start, end=end, interval="1h", limit=1000
)
candle_records.extend(candle_result.data)

while candle_result.next_cursor:
    candle_result = client.hyperliquid.candles.history(
        COIN, start=start, end=end, interval="1h",
        cursor=candle_result.next_cursor, limit=1000
    )
    candle_records.extend(candle_result.data)

candles_df = pd.DataFrame(
    [
        {
            "timestamp": c.timestamp,
            "open": float(c.open),
            "high": float(c.high),
            "low": float(c.low),
            "close": float(c.close),
            "volume": float(c.volume),
        }
        for c in candle_records
    ]
)
candles_df["timestamp"] = pd.to_datetime(candles_df["timestamp"], utc=True)
candles_df = candles_df.sort_values("timestamp").reset_index(drop=True)
print(f"{len(candles_df)} hourly candles loaded")

## 4. Liquidation Heatmap: Price vs Time

This is the core visualization. Each cell shows the total notional value liquidated at a given price level during a time bucket. Bright spots = liquidation clusters.

In [ ]:
# Bin liquidations into time buckets and price buckets
PRICE_BINS = 50
TIME_FREQ = "2h"

df["price_bin"] = pd.cut(df["price"], bins=PRICE_BINS)
df["time_bin"] = df["timestamp"].dt.floor(TIME_FREQ)

heatmap_data = (
    df.groupby(["price_bin", "time_bin"], observed=True)["notional"]
    .sum()
    .unstack(fill_value=0)
)

# Use price bin midpoints as labels
price_labels = [f"${iv.mid:,.0f}" for iv in heatmap_data.index]
time_labels = [t.strftime("%m/%d %H:%M") for t in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(
    heatmap_data.values,
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={"label": "Notional Liquidated ($)"},
    xticklabels=time_labels,
    yticklabels=price_labels,
)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Price Level")
ax.set_title(f"{COIN} Liquidation Heatmap — {LOOKBACK_DAYS}d Window", fontsize=16)
ax.invert_yaxis()

# Reduce label clutter
ax.set_xticks(ax.get_xticks()[::max(1, len(time_labels) // 12)])
ax.set_yticks(ax.get_yticks()[::max(1, len(price_labels) // 20)])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Liquidation Scatter Plot with Price Overlay

Each dot is a liquidation event. Size = notional value, color = long (red) vs short (green). The BTC price is overlaid as a line.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))

# Sample for performance if dataset is large
plot_df = df.sample(n=min(20_000, len(df)), random_state=42) if len(df) > 20_000 else df

colors = plot_df["side"].map({"Long": "#e74c3c", "Short": "#2ecc71"})
sizes = np.clip(plot_df["notional"] / plot_df["notional"].quantile(0.95) * 30, 2, 200)

ax.scatter(
    plot_df["timestamp"], plot_df["price"],
    c=colors, s=sizes, alpha=0.3, edgecolors="none",
)

# Price overlay
ax2 = ax.twinx()
ax2.plot(candles_df["timestamp"], candles_df["close"], color="#f39c12", linewidth=1.5,
         alpha=0.8, label="BTC Price")
ax2.set_ylabel("BTC Price ($)")

# Align y-axes
price_min, price_max = df["price"].min(), df["price"].max()
padding = (price_max - price_min) * 0.05
ax.set_ylim(price_min - padding, price_max + padding)
ax2.set_ylim(price_min - padding, price_max + padding)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Liquidation Price ($)")
ax.set_title(f"{COIN} Liquidation Events — Longs (red) vs Shorts (green)", fontsize=16)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e74c3c", markersize=8, label="Long Liquidation"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ecc71", markersize=8, label="Short Liquidation"),
    Line2D([0], [0], color="#f39c12", linewidth=1.5, label="BTC Price"),
]
ax.legend(handles=legend_elements, loc="upper left")
plt.tight_layout()
plt.show()

## 6. Time-of-Day Analysis

When do liquidations hit hardest? This shows the hourly distribution of liquidation volume.

In [ ]:
hourly = df.groupby(["hour", "side"])["notional"].sum().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar chart
hourly.plot(kind="bar", stacked=True, ax=axes[0],
            color={"Long": "#e74c3c", "Short": "#2ecc71"})
axes[0].set_title("Liquidation Volume by Hour (UTC)", fontsize=14)
axes[0].set_xlabel("Hour of Day (UTC)")
axes[0].set_ylabel("Total Notional ($)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(title="Side")

# Count by hour
hourly_count = df.groupby(["hour", "side"]).size().unstack(fill_value=0)
hourly_count.plot(kind="bar", stacked=True, ax=axes[1],
                  color={"Long": "#e74c3c", "Short": "#2ecc71"})
axes[1].set_title("Liquidation Count by Hour (UTC)", fontsize=14)
axes[1].set_xlabel("Hour of Day (UTC)")
axes[1].set_ylabel("Number of Liquidations")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Side")

plt.tight_layout()
plt.show()

## 7. Liquidation Size Distribution

How big are typical liquidations? This shows the distribution of individual liquidation sizes (notional value).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Log-scale histogram of notional sizes
for side, color in [("Long", "#e74c3c"), ("Short", "#2ecc71")]:
    subset = df[df["side"] == side]["notional"]
    axes[0].hist(subset, bins=100, alpha=0.6, color=color, label=side,
                 log=True, edgecolor="none")
axes[0].set_xlabel("Notional Value ($)")
axes[0].set_ylabel("Count (log scale)")
axes[0].set_title("Liquidation Size Distribution", fontsize=14)
axes[0].legend()
axes[0].set_xlim(0, df["notional"].quantile(0.99))

# Summary stats table
stats = df.groupby("side")["notional"].describe(percentiles=[0.5, 0.75, 0.95, 0.99])
stats_display = stats[["count", "mean", "50%", "75%", "95%", "99%", "max"]].round(0)
axes[1].axis("off")
table = axes[1].table(
    cellText=[[f"{v:,.0f}" for v in row] for row in stats_display.values],
    rowLabels=stats_display.index,
    colLabels=["Count", "Mean $", "Median $", "P75 $", "P95 $", "P99 $", "Max $"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
axes[1].set_title("Notional Value Statistics by Side ($)", fontsize=14, pad=20)

plt.tight_layout()
plt.show()

## 8. Liquidation Cascade Detection

A "cascade" occurs when a burst of liquidations happen in rapid succession, often pushing price further and triggering more liquidations. We detect these by finding time windows with abnormally high liquidation counts.

In [ ]:
# Resample into 5-minute buckets
bucketed = df.set_index("timestamp").resample("5min").agg(
    count=("price", "size"),
    notional=("notional", "sum"),
    avg_price=("price", "mean"),
).dropna()

# Detect cascades: buckets above the 99th percentile of liquidation count
threshold = bucketed["count"].quantile(0.99)
cascades = bucketed[bucketed["count"] >= threshold].copy()
cascades = cascades.sort_values("notional", ascending=False)

print(f"Cascade threshold (P99): {threshold:.0f} liquidations per 5-min bucket")
print(f"Detected {len(cascades)} cascade events\n")

fig, ax = plt.subplots(figsize=(18, 6))

ax.bar(bucketed.index, bucketed["notional"], width=pd.Timedelta("5min"),
       color="#3498db", alpha=0.5, label="Normal")
ax.bar(cascades.index, cascades["notional"], width=pd.Timedelta("5min"),
       color="#e74c3c", alpha=0.9, label="Cascade (P99+)")

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Notional Liquidated ($)")
ax.set_title(f"{COIN} Liquidation Cascades — 5-Minute Buckets", fontsize=16)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Top cascade events
print("\nTop 10 Cascade Events (5-min windows):")
print("-" * 70)
for ts, row in cascades.head(10).iterrows():
    print(f"  {ts:%Y-%m-%d %H:%M} UTC | {row['count']:.0f} liquidations | ${row['notional']:>12,.0f} notional | ~${row['avg_price']:,.0f}")

## 9. Long vs Short Imbalance Over Time

The ratio of long-to-short liquidation volume reveals market sentiment. When longs are getting liquidated heavily, the market is dumping. When shorts get wiped out, it's a squeeze.

In [ ]:
# Resample into hourly buckets by side
hourly_side = (
    df.set_index("timestamp")
    .groupby("side")
    .resample("1h")["notional"]
    .sum()
    .unstack(level=0, fill_value=0)
)

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

# Stacked area
axes[0].fill_between(hourly_side.index, 0, hourly_side.get("Long", 0),
                     color="#e74c3c", alpha=0.6, label="Long Liquidations")
axes[0].fill_between(hourly_side.index, 0, -hourly_side.get("Short", 0),
                     color="#2ecc71", alpha=0.6, label="Short Liquidations")
axes[0].axhline(0, color="white", linewidth=0.5)
axes[0].set_ylabel("Notional ($)")
axes[0].set_title(f"{COIN} Hourly Liquidation Volume — Long vs Short", fontsize=16)
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"${abs(x)/1e6:.1f}M")
)
axes[0].legend(loc="upper left")

# Price subplot
axes[1].plot(candles_df["timestamp"], candles_df["close"], color="#f39c12", linewidth=1.5)
axes[1].set_ylabel("BTC Price ($)")
axes[1].set_xlabel("Time (UTC)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 10. Day-of-Week vs Hour Heatmap

A calendar-style heatmap showing which day/hour combinations see the most liquidation activity.

In [ ]:
df["weekday"] = df["timestamp"].dt.day_name()

day_hour = (
    df.groupby(["weekday", "hour"])["notional"]
    .sum()
    .unstack(fill_value=0)
)

# Reorder days
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_hour = day_hour.reindex([d for d in day_order if d in day_hour.index])

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    day_hour,
    cmap="magma",
    ax=ax,
    fmt=".0f",
    cbar_kws={"label": "Notional Liquidated ($)"},
    linewidths=0.5,
)
ax.set_title(f"{COIN} Liquidation Volume — Day of Week vs Hour (UTC)", fontsize=14)
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 11. Summary Statistics

In [ ]:
print(f"{'='*60}")
print(f"  {COIN} LIQUIDATION SUMMARY — {LOOKBACK_DAYS} Day Window")
print(f"  {start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} UTC")
print(f"{'='*60}")
print(f"  Total liquidations:    {len(df):>12,}")
print(f"  Total notional:        ${df['notional'].sum():>12,.0f}")
print(f"  Avg per hour:          {len(df) / (LOOKBACK_DAYS * 24):>12,.0f}")
print(f"  Price range:           ${df['price'].min():>10,.0f} - ${df['price'].max():,.0f}")
print()

for side in ["Long", "Short"]:
    s = df[df["side"] == side]
    print(f"  {side} liquidations:")
    print(f"    Count:               {len(s):>12,}  ({len(s)/len(df)*100:.1f}%)")
    print(f"    Total notional:      ${s['notional'].sum():>12,.0f}")
    print(f"    Median size:         ${s['notional'].median():>12,.0f}")
    print(f"    Largest single:      ${s['notional'].max():>12,.0f}")
    print()

print(f"  Biggest liquidation hour (UTC): {df.groupby('hour')['notional'].sum().idxmax()}:00")
print(f"  Quietest hour (UTC):            {df.groupby('hour')['notional'].sum().idxmin()}:00")
print(f"{'='*60}")

In [ ]:
client.close()
print("Client closed. Done!")